In [2]:
##### Decision Trees #####
# Let's say we have a dataset with a great number of features.
# Traditional approaches such as linear or logistical regressions are imprecise and cannot and will not evaluate correctly and precisely data at such a scale
# What does decision tree bring new to the table? everything!
# We have the following binary tree from data structures :
#                          root
#                  yes node ---- | ---- no node
#
# Now, how does this help? We "ask a question" : what is the feature that splits data in the best way? What feature has a high correlation to the output we desire?
# Let's take cancer : if we had a feature like name, its pretty much useless. what if we had access to the history of the family checkups and medical records which show the prob of a person to get cancer? That's more interesting!
# This is what decision trees do : they look at all features and choose only the feature that is the most "interesting".
# We can either take log entropy which is better but for a larger scale computes bad values ( due to computer precision being bad ) ORRR weighted gini!
# We basically want to minimise the score that starts off as "infinite" and get closer to 0. aka, we are "sure" that by this split the data is spread more evenly
# Say we have : 1 1 1 0 1 after a split. After evaluating the weighted gini, we can clearly tell that its pretty close to 0. why? 
# This is the formula for gini : 1 - sum prob(class_i)**2.
# We have 5 samples, prob(1) = 4/5, prob(0) = 1/5
# Now, what is the probability of choosing one after choosing one? 4/5 ** 2.
# Gini is based off two extractions, and now we see that 1 - 4/5 ** 2 means that we dont count the cases where we only choose back to back 1
# We also subtract 1/5 **2 to not count cases where we subtract only 0.
# we can now see that this gini is 1 - 17/25 = 8/25, pretty close to 0.
# If we only had 1, it would be 0. aka, there is 0 uncertainty in our choice:) 
# Decision trees, if it were used in a regression task, instead of gini they would be implemented with the MSE of parent or variance of parent
import numpy as np
import pandas as pd
import kagglehub

x, y = None, None
dataset = None
model = None

stoi, vocab = {}, []

def can_be_value(string) -> bool:
    try:
        x = float(string)
    except:
        return False

    return True
    
def encode(string) -> int: 
    global vocab, stoi 
    n = 0
    base = 64
    for x in string:
        n = n * base + stoi[x]
    return n
    
def load_data():
    """
    Data preprocessing function. The functions defined above were used by me to convert non-numerical
    into numerical
    """
    global x, y
    global dataset, stoi, vocab

    dataset = pd.read_csv("/kaggle/input/datasets/yasserh/titanic-dataset/Titanic-Dataset.csv")
    dataset["Sex"] = dataset["Sex"].replace({"female" : 1, "male" : 0})
    dataset["Embarked"] = dataset["Embarked"].replace({"S" : 1, "C" : 2, "Q" : 3})
    
    # process names and attribute for each surname an unique number
    # also store the unique letters to use later on
    chars = sorted(list(dataset["Name"]))
    for i,a in enumerate(chars): 
        chars[i] = set(a) 
        
    for i in (chars): 
        for j in i: 
            vocab.append(j) 
            
    vocab.append("X")
    for i in range(10):
        vocab.append(str(i))

    vocab = sorted(set(vocab)) 
    stoi = {s:i for i,s in enumerate(vocab)}
    # until now we just took the unique letters we found in the names to create an alphabet
    # now we take the surnames and attribute unique values
    dataset["Name"] = dataset["Name"].str.split(",").str[0]
    unique_names = {name: i for i, name in enumerate(dataset["Name"].unique())}
    dataset["Name"] = dataset["Name"].map(unique_names)

    # here we separate the tickets ( i didnt want to use regex cuz im a boss )
    # take first part of tickets and use the unique vocabulary
    temp = []
    for i in dataset["Ticket"]:
        temp.append(i.split(" "))
    
    values = []
    for j in temp:
        for idx, elem in enumerate(j):
            if can_be_value(elem) == False:
                j[idx] = encode(elem)
        values.append(j)
        
    # now we separate into two different columns one with ticket type and second the number
    dataset["Ticket"] = values
    dataset["TicketPrefix"] = dataset["Ticket"].apply(lambda t: t[0] if len(t) == 2 else 0)
    dataset["TicketNumber"] = dataset["Ticket"].apply(lambda t: t[-1])
    dataset = dataset.drop(columns=["Ticket"])

    # again no regex cuz im a boss, here we take the first letter and convert according to second dictionary and fill NaN with 0
    stoi2 = {s:i+1 for i,s in enumerate("ABCDEFGT")}
    dataset["Cabin"] = dataset["Cabin"].fillna(0)
    dataset["CabinDeck"] = dataset["Cabin"].apply(lambda t: stoi2[t[0]] if isinstance(t, str) == True else 0)

    # now we want the numbers of first cabin only but first we memorise all cabins
    temp = []
    for i in dataset["Cabin"]:
        if i == 0:
            temp.append(0)
            continue
        temp.append(i.split(" "))

    # here we take only first cabins
    for idx, elem in enumerate(temp):
        if isinstance(elem, int) == True:
            continue
        if len(elem) != 1:
            temp[idx] = elem[0]

    # convert from list of lists to list and take only numbers
    temp = [0 if isinstance(i, int) == True else i[0] for i in temp]
    for idx, elem in enumerate(temp):
        if isinstance(elem, int):
            continue
        digits = elem[1:]
        temp[idx] = int(digits) if digits else 0

    dataset["CabinNumber"] = temp
    dataset = dataset.drop(columns=["Cabin"])
    dataset["Age"] = dataset["Age"].fillna(dataset["Age"].median())
    dataset["Embarked"] = dataset["Embarked"].fillna(1)
    
    x = []
    for elem in dataset:
        if elem == "Survived":
            continue
        x.append(dataset[elem].to_numpy(dtype=float))
    
    x = np.array(x)
    y = np.array(list(dataset["Survived"]))
    
load_data()

class Node():
    """
    Helper Node class, used to check whether it's a split node or a leaf node
    """
    def __init__(self, left=None, right=None, feature=None, value=None, threshold=None):
        self.left = left
        self.right = right
        self.value = value
        self.feature = feature
        self.threshold = threshold

    
class DecisionTree():
    """
    Decision Tree class, from scratch
    """
    def __init__(self, max_depth=10, min_samples_split=2): #max 10 lvls, a split can only occur if we have at least 2 samples
        self.max_depth = max_depth
        self.root = None
        self.min_s_s = min_samples_split

    def _gini(self, y): # compute the confidence of choosing a desired element from distinct classes ; how sure we are in our choice that it is what we wanted
        unique, counts = np.unique(y, return_counts=True)
        
        if len(unique) == 1: # we have 1 prob since we only have one class and 1-1=0
            return 0

        probs = counts/np.sum(counts) # u basically count the cases where u extract an instance of each class

        return 1-np.sum(probs**2) # take [1, 1, 0] : if u were to choose with return 1 twice, it would be (2/3)^2. hence, we have 1 - sum pi**2

    def _split(self, x_row, threshold): # we simply look in our samples where the values are under/ over thresholds.
        left_indices = np.where(x_row < threshold)[0]
        right_indices = np.where(x_row >= threshold)[0]
        
        return left_indices, right_indices

    def _majority_class(self, y): # we just take from a class the element that occurs the most
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]

    def _candidate_thresholds(self, x_row): # we take all possible thresholds
        candidates = np.unique(x_row)
        for idx in range(len(candidates) - 1):
            candidates[idx] = (candidates[idx] + candidates[idx+1])/2

        return candidates[:-1]

    def _best_split(self, x, y):
        best_score = float('inf') # we start with an infinite score, our goal is to minimise it
        feat, tresh = None, None
        l, r = None, None
        m = x.shape[0]
        n = x.shape[1]
        for feature in range(m): # for each row, aka each feature(theoretically it should be inverse, features = columns but oh well)
            candidates = self._candidate_thresholds(x[feature]) # self explanatory from here on
            for threshold in candidates:
                left, right = self._split(x[feature], threshold)
                if len(left) == 0 or len(right) == 0:
                    continue

                g_left = self._gini(y[left])
                g_right = self._gini(y[right])

                weighted_gini = (len(left)/n) * g_left + (len(right)/n) * g_right # the only thing we do here is a weighted gini to determine if we have improved
                
                if weighted_gini < best_score:
                    best_score = weighted_gini
                    feat = feature
                    tresh = threshold
                    l = left
                    r = right
                    
        return (feat, tresh, l, r)

    def _grow_tree(self, x, y, depth=0):

        if len(np.unique(y)) == 1: # if y only has survivors or no survivors
            return Node(value=y[0])

        if depth >= self.max_depth: # if we have reached the "bottom"
            return Node(value=self._majority_class(y))

        if len(y) < self.min_s_s: # if we have the number of samples lower than the minimum
            return Node(value=self._majority_class(y))

        f, t, l, r = self._best_split(x, y)
        if f is None: # if we cannot do a split aka the data cannot be distinguishable
            return Node(value=self._majority_class(y))

        
        node = Node(feature=f, threshold = t)

        node.left = self._grow_tree(x[:, l], y[l], depth + 1)
        node.right = self._grow_tree(x[:, r], y[r], depth + 1)

        return node

    def fit(self, x, y):
        self.root = self._grow_tree(x, y)

    def _traverse_tree(self, sample, node): # simple function that goes thru the tree
        if node.value is not None: # if leaf
            return node.value
    
        
        if sample[node.feature] < node.threshold: # if we go left or right
            return self._traverse_tree(sample, node.left)
    
        return self._traverse_tree(sample, node.right)

    def predict(self, x):

        predictions = []

        for idx in range(x.shape[1]): # we take each example and store its "run" in our decision tree
            sample = x[:, idx]
            predictions.append(self._traverse_tree(sample, self.root))

        return np.array(predictions)

tree = DecisionTree(
    max_depth=10,
    min_samples_split=2
)

tree.fit(x, y)

pred = tree.predict(x)

accuracy = np.mean(pred == y)

print(f"Accuracy: {accuracy:.4f}")

##### Use cases #####
# well defined rules, properties, easy splits, dependent features
# great when we have a relatively larger set of features
# does not depend on linearity

/tmp/ipykernel_58/2260661213.py:32: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["Sex"] = dataset["Sex"].replace({"female" : 1, "male" : 0})
/tmp/ipykernel_58/2260661213.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["Embarked"] = dataset["Embarked"].replace({"S" : 1, "C" : 2, "Q" : 3})


Accuracy: 0.9517
